## Проверка состояния кластера OpenSearch и списка индексов через API

In [26]:
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = "https://127.0.0.1:9200/_cluster/health"
auth = ("admin", "SamplePassword1!")

response = requests.get(url, auth=auth, verify=False)
data = response.json()

print("Cluster status:", data["status"])
print("Nodes:", data["number_of_nodes"])
print("Active shards:", data["active_shards"])

Cluster status: yellow
Nodes: 1
Active shards: 31


In [24]:
import requests

url = "https://127.0.0.1:9200/_cat/indices?v"
auth = ("admin", "SamplePassword1!")

response = requests.get(url, auth=auth, verify=False)

print(response.text)

health status index                               uuid                   pri rep docs.count docs.deleted store.size pri.store.size
yellow open   security-auditlog-2026.04.03        nSjLtSJhSE2o5Q7dlVpAcQ   1   1         23            0    116.3kb        116.3kb
green  open   .ql-datasources                     x-qLMcNdSFeHKVFWrGkrHg   1   0          0            0       208b           208b
green  open   .kibana_1563580327_teachertenant_1  O3Bx9Tt3QWmPQLGMtGfCEg   1   0         18            0     21.7kb         21.7kb
green  open   .kibana_1876018518_student1_1       moe-4lmqTAuC2fxfbdxy9w   1   0          1            0      5.3kb          5.3kb
yellow open   security-auditlog-2026.03.30        mXcSHGL3SDORh8ttR7MzZw   1   1          1            0     16.6kb         16.6kb
green  open   .opendistro_security                wHNhzzfsRGO3beY0ZhJqNg   1   0         10            2    281.8kb        281.8kb
yellow open   security-auditlog-2026.04.22        W2gvbrpCQ--N-HWTQV4e1A   1   1   

/opt/anaconda3/lib/python3.12/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '127.0.0.1'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


## Скрипт пакетной загрузки данных о студенческой активности в OpenSearch

In [30]:
import pandas as pd
import json
import requests
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = "https://127.0.0.1:9200/_bulk"
auth = ("admin", "SamplePassword1!")
headers = {"Content-Type": "application/json"}


def load_csv_to_opensearch(csv_file, index_name, batch_size=1000):

    print(f"Loading {csv_file} → {index_name}")

    df = pd.read_csv(csv_file)
    df = df.fillna("")

    total = len(df)
    print("Total rows:", total)

    for start in range(0, total, batch_size):

        end = start + batch_size
        batch = df.iloc[start:end]

        bulk_data = ""

        for _, row in batch.iterrows():
            bulk_data += json.dumps({"index": {"_index": index_name}}) + "\n"
            bulk_data += json.dumps(row.to_dict(), ensure_ascii=False) + "\n"

        response = requests.post(
            url,
            headers=headers,
            data=bulk_data.encode("utf-8"),
            auth=auth,
            verify=False
        )

        print(f"Loaded {end if end < total else total}/{total} | status {response.status_code}")


# загрузка файлов
load_csv_to_opensearch("assessments.csv", "assessments")
load_csv_to_opensearch("courses.csv", "courses")
load_csv_to_opensearch("vle.csv", "vle")
load_csv_to_opensearch("studentAssessment.csv", "student_assessment")
load_csv_to_opensearch("studentInfo.csv", "student_info")
load_csv_to_opensearch("studentRegistration.csv", "student_registration")
load_csv_to_opensearch("studentVle.csv", "student_vle")

Loading studentVle.csv → student_vle
Total rows: 10655280
Loaded 1000/10655280 | status 200
Loaded 2000/10655280 | status 200
Loaded 3000/10655280 | status 200
Loaded 4000/10655280 | status 200
Loaded 5000/10655280 | status 200
Loaded 6000/10655280 | status 200
Loaded 7000/10655280 | status 200
Loaded 8000/10655280 | status 200
Loaded 9000/10655280 | status 200
Loaded 10000/10655280 | status 200
Loaded 11000/10655280 | status 200
Loaded 12000/10655280 | status 200
Loaded 13000/10655280 | status 200
Loaded 14000/10655280 | status 200
Loaded 15000/10655280 | status 200
Loaded 16000/10655280 | status 200
Loaded 17000/10655280 | status 200
Loaded 18000/10655280 | status 200
Loaded 19000/10655280 | status 200
Loaded 20000/10655280 | status 200
Loaded 21000/10655280 | status 200
Loaded 22000/10655280 | status 200
Loaded 23000/10655280 | status 200
Loaded 24000/10655280 | status 200
Loaded 25000/10655280 | status 200
Loaded 26000/10655280 | status 200
Loaded 27000/10655280 | status 200
Loaded

извлечь данные из opensearch

In [37]:
import requests
import pandas as pd

url = "https://127.0.0.1:9200/student_info/_search"
auth = ("admin","SamplePassword1!")

query = {
    "size": 10000,
    "query": {"match_all": {}}
}

response = requests.get(url, json=query, auth=auth, verify=False)

data = response.json()

rows = [doc["_source"] for doc in data["hits"]["hits"]]

df = pd.DataFrame(rows)

df.head(100)

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass
...,...,...,...,...,...,...,...,...,...,...,...,...
95,AAA,2013J,236284,M,Scotland,Post Graduate Qualification,90-100%,0-35,0,60,N,Pass
96,AAA,2013J,238007,M,South Region,HE Qualification,90-100%,55<=,0,60,N,Pass
97,AAA,2013J,240712,M,London Region,HE Qualification,80-90%,0-35,0,80,N,Pass
98,AAA,2013J,240884,M,South Region,A Level or Equivalent,90-100%,0-35,0,120,N,Pass
